In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 6.29 Scattering in Three Dimensions: Partial Waves and the Born Approximation

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VI — Quantum Mechanics",
    number="6.29",
    title="Scattering in Three Dimensions: Partial Waves and the Born " "Approximation",
    blurb="How we actually learn what matter is made of: a beam, a target, "
    "and the angles things come out at. The scattering amplitude built "
    "wave by partial wave, phase shifts extracted from a radial "
    "integration and certified against exact results, the optical "
    "theorem verified to eight decimals, Born's approximation caught "
    "working and caught failing, a resonance fit to its Breit–Wigner "
    "shape, and the quantum transparency that lets slow electrons "
    "slip through argon unseen.",
    difficulty="advanced",
    estimate="150–190 min",
)

## Notebook overview

Nearly everything we know about the subatomic world arrived by
scattering. Rutherford discovered the nucleus by counting deflected
alpha particles ([§2.5](../02-classical-mechanics/scattering.ipynb)
told the classical version of that story); electron scattering mapped
the proton's charge distribution; neutron scattering is how we see
inside materials today. [§6.13](scattering-tunneling.ipynb) treated
scattering in one dimension, where there are only two directions to
go. This notebook does it properly, in three, where the question
becomes an *angular* one — how much goes *where* — and the answer is
organized by the beautiful machinery of **partial waves**: decompose
the beam into angular-momentum components, note that a spherically
symmetric target cannot mix them, and reduce the entire
three-dimensional problem to one number per channel, the phase shift
$\delta_\ell$.

The plan mirrors the course's standing method: build the machinery
from scratch, certify it against every exact result we can reach
(hard sphere, square well), and only then aim it at physics it was
not calibrated on — Born's approximation and its domain of validity,
a sharp resonance with its Breit–Wigner profile, and the
Ramsauer–Townsend transparency that stumped classical physics in
1921. Griffiths ch. 11 {cite}`griffiths_qm` is the companion text.

A note on reading the checks in this notebook: a validation compares a
result to an expected physical fact. A ✗ does not by itself mean the
answer is wrong; it means the output did not match what the check
expected, which may be a genuine error, a different-but-valid
convention, or too tight a tolerance. Treat a ✗ as a prompt to locate
the discrepancy. Passing is strong evidence, not proof.

## Theory in brief

**The scattering boundary condition.** A steady beam plus an outgoing
ripple: far from the target the stationary state must look like

```{math}
:label: eq-sc3-asym
\psi(\mathbf r) \;\longrightarrow\;
e^{ikz} \;+\; f(\theta)\,\frac{e^{ikr}}{r},
```

and the **scattering amplitude** $f(\theta)$ carries all the physics:
the differential cross-section is $d\sigma/d\Omega = |f(\theta)|^2$.
This is the three-dimensional sibling of
[§6.13](scattering-tunneling.ipynb)'s reflection amplitude, with an
angle where 1D had only "backwards."

**Partial waves.** For a central potential, angular momentum is
conserved, so each $\ell$ scatters independently
([§6.16](central-potentials-3d.ipynb)'s radial equation, once per
channel). A potential of finite range can do exactly one thing to the
$\ell$-th radial wave: shift its asymptotic phase by $\delta_\ell$.
Those shifts assemble the amplitude and the total cross-section,

```{math}
:label: eq-sc3-pw
f(\theta) = \frac{1}{k}\sum_{\ell=0}^{\infty} (2\ell+1)\,
e^{i\delta_\ell}\sin\delta_\ell\, P_\ell(\cos\theta),
\qquad
\sigma = \frac{4\pi}{k^2}\sum_{\ell=0}^{\infty}(2\ell+1)
\sin^2\delta_\ell ,
```

and the sum is short: classically, angular momentum $\ell$ means
impact parameter $\ell/k$, so only $\ell \lesssim ka$ can feel a
target of range $a$ — the same semiclassical bookkeeping that
organized [§2.5](../02-classical-mechanics/scattering.ipynb).

**The optical theorem.** Unitarity — scattered flux has to come from
somewhere, namely the shadow carved out of the forward beam — forces
an exact identity between the total cross-section and the *forward*
amplitude,

```{math}
:label: eq-sc3-optical
\sigma \;=\; \frac{4\pi}{k}\,\mathrm{Im}\, f(0),
```

which we will verify to eight decimal places: a stringent
self-consistency test that any correct scattering computation must
pass, and any buggy one will fail loudly.

**The Born approximation.** When the potential is weak, the wave is
barely disturbed, and first-order perturbation theory (the same
apparatus as [§6.24](time-dependent-perturbation.ipynb)'s golden
rule) gives the amplitude as a Fourier transform of the potential:
with $\hbar = m = 1$,

```{math}
:label: eq-sc3-born
f_{\rm B}(\theta) = -\frac{1}{2\pi}\!\int d^3r\;
e^{i\mathbf q\cdot\mathbf r}\, V(\mathbf r)
\;\;\xrightarrow{\;\text{Yukawa } -V_0 e^{-\mu r}/r\;}\;\;
\frac{2V_0}{\mu^2 + q^2},
\qquad q = 2k\sin\tfrac{\theta}{2},
```

and in the $\mu \to 0$ limit $|f_{\rm B}|^2$ becomes *exactly* the
Rutherford cross-section of
[§2.5](../02-classical-mechanics/scattering.ipynb) — one of quantum
mechanics' small miracles. Born is an approximation, though, and this
notebook measures where it breaks.

**Resonances.** When the potential can almost bind a state behind its
centrifugal barrier, the phase shift races through $\pi/2$ over a
narrow energy window and the channel's cross-section spikes to its
unitarity maximum with the universal profile

```{math}
:label: eq-sc3-bw
\sin^2\delta_\ell \;=\;
\frac{(\Gamma/2)^2}{(E - E_R)^2 + (\Gamma/2)^2}
```

— the Breit–Wigner shape {cite}`breit1936`, the lineshape of every
particle-physics bump and every nuclear resonance ever tabulated.

Units: $\hbar = m = 1$ throughout, so $E = k^2/2$; the potential
range sets the length unit ($a = 1$).

## Setup

The workhorse is a phase-shift extractor: integrate the radial
equation for $u_\ell(r)$ outward with `scipy.integrate.solve_ivp`,
then match to the free asymptotic form
$u_\ell \propto kr[\cos\delta_\ell\, j_\ell(kr) -
\sin\delta_\ell\, y_\ell(kr)]$ at two radii beyond the potential's
range. Everything is deterministic.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import quad, solve_ivp
from scipy.optimize import brentq, curve_fit
from scipy.special import spherical_jn, spherical_yn

from ecp import validate


def phase_shift(V, l, k, r_start, u0, up0, r_match=25.0):
    """Phase shift δ_ℓ from an outward radial integration.

    Integrates u'' = [ℓ(ℓ+1)/r² + 2V(r) − k²] u from r_start with the
    given initial data, then matches u/r at two radii (r_match − 1.3
    and r_match) to the free combination cos δ · j_ℓ − sin δ · y_ℓ.
    Returns δ_ℓ in the arctan branch (−π/2, π/2]; sin²δ_ℓ, which is
    all the cross-section needs, is branch-independent.

    Parameters
    ----------
    V : callable
        Central potential V(r) (must be negligible beyond r_match − 2).
    l : int
        Angular momentum quantum number ℓ.
    k : float
        Wavenumber (E = k²/2 with ħ = m = 1).
    r_start : float
        Starting radius of the integration.
    u0, up0 : float
        Initial values u(r_start) and u'(r_start).
    r_match : float
        Outer matching radius.

    Returns
    -------
    float
        The phase shift δ_ℓ.
    """

    def rhs(r, y):
        u, up = y
        return [up, (l * (l + 1) / r**2 + 2.0 * V(r) - k**2) * u]

    sol = solve_ivp(
        rhs, (r_start, r_match), [u0, up0], rtol=1e-9, atol=1e-13, dense_output=True
    )
    r1, r2 = r_match - 1.3, r_match
    u1, u2 = sol.sol(r1)[0], sol.sol(r2)[0]
    j1, y1 = spherical_jn(l, k * r1), spherical_yn(l, k * r1)
    j2, y2 = spherical_jn(l, k * r2), spherical_yn(l, k * r2)
    g = (u1 * r2) / (u2 * r1)
    return float(np.arctan((g * j2 - j1) / (g * y2 - y1)))


def phase_soft(V, l, k, r0=0.05, r_match=25.0):
    """Phase shift for a potential regular enough to start near r = 0.

    Launches the integration at small r0 with the centrifugal series
    behaviour u ~ r^(ℓ+1), which is exact for any potential less
    singular than 1/r² at the origin.

    Parameters
    ----------
    V : callable
        Central potential V(r).
    l : int
        Angular momentum ℓ.
    k : float
        Wavenumber.
    r0 : float
        Launch radius.
    r_match : float
        Outer matching radius.

    Returns
    -------
    float
        The phase shift δ_ℓ.
    """
    return phase_shift(V, l, k, r0, r0 ** (l + 1), (l + 1) * r0**l, r_match)


def sigma_l(l, k, delta):
    """Partial cross-section (4π/k²)(2ℓ+1) sin²δ_ℓ of one channel.

    Parameters
    ----------
    l : int
        Angular momentum ℓ.
    k : float
        Wavenumber.
    delta : float
        Phase shift δ_ℓ.

    Returns
    -------
    float
        The channel's contribution to the total cross-section.
    """
    return 4.0 * np.pi / k**2 * (2 * l + 1) * np.sin(delta) ** 2


def f_amplitude(theta, k, deltas):
    """Scattering amplitude f(θ) from a list of phase shifts.

    Sums (2ℓ+1) e^(iδ_ℓ) sin δ_ℓ P_ℓ(cos θ)/k over the channels
    provided, using numpy's Legendre evaluation for P_ℓ.

    Parameters
    ----------
    theta : float
        Scattering angle in radians.
    k : float
        Wavenumber.
    deltas : sequence of float
        Phase shifts δ_0, δ_1, … .

    Returns
    -------
    complex
        The amplitude f(θ).
    """
    total = 0.0 + 0.0j
    for l, d in enumerate(deltas):
        p_l = np.polynomial.legendre.legval(np.cos(theta), [0.0] * l + [1.0])
        total += (2 * l + 1) * np.exp(1j * d) * np.sin(d) * p_l
    return total / k

## Exercise 1 — The hard sphere: a cross-section is not an area

The impenetrable sphere of radius $a = 1$ is the one target whose
phase shifts are exactly known: the wave must vanish at the wall, so
$\tan\delta_\ell = j_\ell(ka)/y_\ell(ka)$.

**Part a)** Verify the s-wave shift is exactly $\delta_0 = -ka$
(modulo $\pi$; check $\tan\delta_0 = \tan(-ka)$ to `atol=1e-12` at
$ka = 0.3$): at low energy the sphere simply pushes the wave out by
its radius.

**Part b)** Sum {eq}`eq-sc3-pw` (60 channels is plenty) across
$ka = 0.1$ to $10$ and verify the two famous limits: at $ka = 0.1$,
$\sigma/\pi a^2 = 3.9868$ (`rtol=1e-3`) — approaching **4**, four
times the geometric cross-section, because a slow quantum wave feels
the whole sphere's surface, not its silhouette; and at $ka = 10$,
$\sigma/\pi a^2 = 2.3972$ (`rtol=1e-3`), descending slowly toward
the short-wavelength limit of **2** — still *twice* geometric,
because diffraction into the shadow counts as scattering too. Verify
also the semiclassical cutoff: at $ka = 5$ the channels
$\ell \le 7$ carry more than $99\%$ of the sum.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    tan_check < 1e-12,
    "the s-wave hard-sphere shift is exactly −ka: the sphere pushes "
    "the slow wave out by its own radius",
    f"tan mismatch {tan_check:.1e}",
)
validate.close(
    np.array([sig_low, sig_high]),
    np.array([3.9868, 2.3972]),
    "and the cross-section runs from 4πa² (slow: the wave feels the "
    "whole surface) toward 2πa² (fast: silhouette + shadow "
    "diffraction) — never the classical πa²",
    rtol=1e-3,
)
validate.check(
    frac_7 > 0.99,
    "the partial-wave sum is short, as ℓ ≲ ka promises: at ka = 5, "
    "eight channels carry over 99% of it",
    f"l ≤ 7 fraction {frac_7:.4f}",
)

## Exercise 2 — The extractor, certified

Now the numerical workhorse that frees us from exactly solvable
targets. `phase_shift` integrates the radial equation of
[§6.16](central-potentials-3d.ipynb) outward with
`scipy.integrate.solve_ivp` and reads $\delta_\ell$ off the
asymptotic matching. Certify it three ways before trusting it
anywhere new.

**Part a)** Hard sphere: launch *from the wall* ($u(a) = 0$,
$u'(a) = 1$) and verify the extracted $\delta_0, \delta_1, \delta_2$
at $k = 1$ match the analytic `delta_hard` values to `atol=1e-6`.

**Part b)** Attractive square well ($V_0 = 2$, $a = 1$): the s-wave
shift has the closed form $\delta_0 = -ka +
\arctan[(k/k')\tan k'a]$ with $k' = \sqrt{k^2 + 2V_0}$. Verify the
numerical extraction at $k = 0.6$ agrees to `atol=1e-5`.

**Part c)** The scattering length — the single number that summarizes
low-energy scattering (and rules ultracold-atom physics): from
$\delta_0 \to -k a_s$ as $k \to 0$, extract
$a_s = -\delta_0(k)/k$ at $k = 0.01$ and verify it against the
closed form $a_s = a - \tan(\sqrt{2V_0}\,a)/\sqrt{2V_0} = 2.0925$
(`rtol=1e-3`). A well of radius $1$ scattering like a hard sphere of
radius $2.09$: the potential's *effect*, not its size.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    max(abs(dn - da) for dn, da in hard_pairs) < 1e-6,
    "the radial integration reproduces the analytic hard-sphere "
    "shifts channel by channel",
    f"worst gap {max(abs(dn - da) for dn, da in hard_pairs):.1e}",
)
validate.check(
    abs(d_well_num - d_well_ana) < 1e-5,
    "and the square well's closed-form s-wave shift, on a target it "
    "was never tuned to",
    f"gap {abs(d_well_num - d_well_ana):.1e}",
)
validate.close(
    np.array([a_s]),
    np.array([a_s_ana]),
    "the scattering length emerges from δ₀ → −k a_s: a radius-1 well "
    "that scatters like a radius-2.09 hard sphere",
    rtol=1e-3,
)

## Exercise 3 — The amplitude and the optical theorem

{eq}`eq-sc3-optical` is unitarity in one line, and because it relates
three *independently computable* quantities, it is the sharpest
self-test in scattering theory.

**Part a)** For the hard sphere at $k = 1$ (40 channels), compute the
total cross-section three ways: the partial-wave sum
{eq}`eq-sc3-pw`, the forward-amplitude form
$(4\pi/k)\,\mathrm{Im}f(0)$, and the direct angular integral
$\int |f(\theta)|^2 \, d\Omega$ with `scipy.integrate.quad`. Verify
the first two agree to `rtol=1e-8` and the integral to `rtol=1e-6`:
eight decimal places of unitarity.

**Part b)** Plot $d\sigma/d\Omega = |f(\theta)|^2$ at $ka = 1$ and
$ka = 5$ and verify the emergence of the forward diffraction peak:
the forward-to-backward ratio $|f(0)|^2/|f(\pi)|^2$ grows from
about $6$ at $ka = 1$ to above $50$ at $ka = 5$ — the shadow's edge
turning into a beam, the wave version of what
[§4.9](../04-special-relativity/relativistic-optics.ipynb) called a
headlight.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    np.array([sig_fwd]),
    np.array([sig_sum]),
    "the optical theorem holds to eight decimals: the whole "
    "cross-section lives in the shadow the forward amplitude carves "
    "from the beam",
    rtol=1e-8,
)
validate.close(
    np.array([sig_int]),
    np.array([sig_sum]),
    "and the angular integral of |f|² closes the triangle: three "
    "independent routes, one number",
    rtol=1e-6,
)
validate.check(
    ratios[1.0] < 10.0 and ratios[5.0] > 50.0,
    "the diffraction peak sharpens with ka: mildly forward at long "
    "wavelength, a fifty-fold forward beam at ka = 5",
    f"f/b ratios {ratios[1.0]:.2f} and {ratios[5.0]:.1f}",
)

## Exercise 4 — Born: an approximation caught in the act

{eq}`eq-sc3-born` is the workhorse of scattering analysis — and it is
first-order perturbation theory, valid only when the wave is barely
disturbed. The Yukawa potential $V = -V_0 e^{-\mu r}/r$ (with
$\mu = 1$, $k = 1$) has a clean Born cross-section to integrate and
an exact answer our certified extractor can produce, so for once we
can put an approximation on trial with full evidence.

**Part a)** Weak coupling, $V_0 = 0.02$: integrate Born's
$|f_{\rm B}|^2$ over solid angle with `scipy.integrate.quad` and
verify it agrees with the exact result — twelve channels of the
certified `phase_soft` extractor summed through {eq}`eq-sc3-pw` —
to within $1\%$ (measured ratio $0.997$).

**Part b)** Strong coupling, $V_0 = 2$ — strong enough to bind
states: verify the same Born integral now *overestimates* the exact
partial-wave cross-section by more than $50\%$ (measured ratio
$1.83$). Born knows nothing of
unitarity: nothing in {eq}`eq-sc3-born` stops $\sin\delta_\ell$
from exceeding one, and at strong coupling it effectively does.
Tabulate the ratio at $V_0 = 0.02, 0.5, 2$ and verify it drifts
monotonically away from $1$.

**Part c)** The Rutherford miracle: verify *algebraically on a grid*
that at $\mu = 0$ the Born cross-section
$|2V_0/q^2|^2$ equals the classical Rutherford formula
$(V_0/4E)^2 \sin^{-4}(\theta/2)$ of
[§2.5](../02-classical-mechanics/scattering.ipynb) at every angle
(`rtol=1e-12`). Quantum perturbation theory, first order, reproduces
Rutherford's classical result exactly — the coincidence that made
the 1911 analysis of the nucleus come out right for reasons
Rutherford could not have known.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    abs(ratios_born[0.02] - 1.0) < 0.01,
    "at weak coupling Born agrees with the exact cross-section to "
    "1%: the wave is barely disturbed, and first order is enough",
    f"ratio {ratios_born[0.02]:.4f}",
)
validate.check(
    ratios_born[2.0] > 1.5,
    "at strong coupling Born overshoots by more than 50%: it knows "
    "nothing of unitarity",
    f"ratio {ratios_born[2.0]:.4f}",
)
validate.check(
    abs(ratios_born[0.02] - 1.0)
    < abs(ratios_born[0.5] - 1.0)
    < abs(ratios_born[2.0] - 1.0),
    "and the error grows monotonically with the coupling: an "
    "approximation with a visible domain of validity",
    f"|ratio − 1| = {abs(ratios_born[0.02] - 1):.4f}, "
    f"{abs(ratios_born[0.5] - 1):.4f}, {abs(ratios_born[2.0] - 1):.4f}",
)
validate.check(
    ruth_gap < 1e-12,
    "the μ → 0 Born cross-section IS Rutherford's formula, angle by "
    "angle: first-order quantum theory lands exactly on the "
    "classical result of §2.5",
    f"max gap {ruth_gap:.1e}",
)

## Exercise 5 — A resonance and its Breit–Wigner shape

Give the square well depth $V_0 = 4.7$ — just shy of the
$V_0 = \pi^2/2 \approx 4.93$ at which it would bind a p-wave state —
and the almost-bound state survives as a **resonance**: trapped for a
while behind the $\ell = 1$ centrifugal barrier before leaking out.

**Part a)** Scan $\delta_1(k)$ from $k = 0.05$ to $1.6$ (the
extractor, 220 points; unwrap the arctan branch with
`numpy.unwrap`). Verify the resonance signature: $\delta_1$ climbs
through $\pi/2$ near $k \approx 0.41$ with slope
$d\delta_1/dk > 10$ — locate the crossing with
`scipy.optimize.brentq` and verify $E_R = k_R^2/2$ lies in
$[0.06, 0.10]$.

**Part b)** Fit the phase itself to the resonant form with a
background, $\delta_1(E) = \delta_{\rm bg} +
\mathrm{arctan2}(\Gamma/2,\, E_R - E)$ — the arctangent step of
{eq}`eq-sc3-bw` riding on the slowly varying phase the well
produces anyway — with `scipy.optimize.curve_fit` in a window of
$\pm 0.04$ around the crossing. Verify the fit is excellent (max
residual below $0.05$ rad), the width obeys
$0.02 < \Gamma < 0.08$, and — the subtlety worth the price of the
exercise — the fitted pole $E_R$ sits *below* the $\pi/2$-crossing:
the negative background phase delays the crossing, so "the energy
where $\delta = \pi/2$" and "the resonance pole" are different
numbers. Verify they reconcile exactly: the energy where the
*fitted model* passes $\pi/2$ reproduces the observed crossing to
`rtol=0.02`. Every "new particle" bump ever announced is this
figure with different axis labels — background phase included.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    0.06 < E_r_cross < 0.10 and steep > 10.0,
    "δ₁ races through π/2 near E ≈ 0.08 with slope > 10: the "
    "almost-bound p state, resonating behind its centrifugal barrier",
    f"E_R = {E_r_cross:.4f}, max slope {steep:.1f}",
)
validate.check(
    0.02 < gamma_fit < 0.08 and resid < 0.05,
    "the Breit–Wigner step with a background fits the phase to "
    "hundredths of a radian: the universal lineshape, measured from "
    "our own phase shifts",
    f"Γ = {gamma_fit:.4f}, max residual {resid:.4f}",
)
validate.check(
    E_r_fit < E_r_cross and d_bg_fit < 0.0,
    "the fitted pole sits BELOW the π/2 crossing because the "
    "negative background phase delays it: pole and crossing are "
    "different numbers",
    f"E_R = {E_r_fit:.4f} vs crossing {E_r_cross:.4f}, " f"δ_bg = {d_bg_fit:+.3f}",
)
validate.close(
    np.array([E_cross_model]),
    np.array([E_r_cross]),
    "and the two reconcile exactly: the fitted model's own π/2 "
    "crossing lands on the observed one",
    rtol=0.02,
)

## Exercise 6 — Ramsauer–Townsend: the invisible atom

In 1921 Ramsauer measured electron–argon cross-sections and found
something impossible: at about $1$ eV, argon is nearly *transparent*
— slow electrons pass through as if the atom were not there
{cite}`ramsauer1921`. No classical picture allows it. Partial waves
explain it in one line: if the well happens to pull the s-wave
forward by *exactly* $\pi$, then $\sin^2\delta_0 = 0$ — a full
wavelength of accumulated phase leaves no trace — and at low energy
there are no other channels to scatter.

**Part a)** For the square well with $V_0 = 11$ — just shy of
binding its *second* s state at $V_0 = (3\pi/2)^2/2 \approx 11.10$ —
build the *continuous* $\delta_0(k)$ from the closed form of
Exercise 2 (track the branch with `numpy.unwrap` from high $k$,
where $\delta_0 \to 0$, down toward threshold) and verify two
things at once by fitting $\delta_0 \approx \pi - a_s k$ over
$k < 0.05$ with `numpy.polyfit`: the intercept is $\pi$ —
**Levinson's theorem**, $\delta_0(0) = n_b\pi$ with $n_b = 1$ bound
state (`atol=0.02`) — and the slope gives a *giant negative*
scattering length, matching the closed form
$a_s = 1 - \tan\sqrt{2V_0}/\sqrt{2V_0} = -8.70$ to `rtol=0.07`
(the few-percent residue is the effective-range correction, which a
scattering length this large drags into view even at $k = 0.05$).
An almost-bound second state makes the well scatter, at threshold,
like a hard sphere eight times its size: the same
near-threshold amplification that Feshbach tuning exploits in cold
atoms.

**Part b)** Find the transparency: the phase, having risen *above*
$\pi$, comes back down through it at a finite $k^*$ — locate the
recrossing with `scipy.optimize.brentq` and verify the s channel
switches off *exactly* there: $\sin^2\delta_0(k^*) < 10^{-12}$,
while at $k^* - 1$ the same channel still scatters with
$\sigma_0 > 1$ and on the far side ($k^* + 0.4$) it has already
recovered by many orders of magnitude — a full wavelength of
accumulated phase leaves the outgoing s wave indistinguishable
from a free one, at one precise energy. Verify separately that the well's *total*
cross-section (channels $\ell \le 5$) still collapses more than
$10\times$ from its low-energy peak. One honest caveat, visible in
the figure: for a square well of unit range the recrossing lands at
$k^*a \approx 2.6$, where $\ell \ge 1$ waves already scatter, so
the transparency is channel-resolved rather than total. Argon's
deeper, atom-shaped potential places its recrossing where s waves
still rule the beam — which is why Ramsauer saw the minimum in the
raw data, and why it is deep rather than zero (the small
$\ell \ge 1$ residue survives there too).

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    np.array([float(intercept)]),
    np.array([np.pi]),
    "Levinson's theorem: one bound s state means δ₀(0) = π exactly — "
    "the phase shift counts the spectrum",
    rtol=0.0,
    atol=0.02,
)
validate.close(
    np.array([a_s_rt]),
    np.array([a_s_rt_ana]),
    "and the threshold slope is a scattering length near −8.7: a "
    "radius-1 well on the verge of a new bound state scatters like a "
    "sphere over eight times its size (the residual is the "
    "effective-range correction)",
    rtol=0.07,
)
validate.check(
    sin2_star < 1e-12
    and sig0_shoulders[0] > 1.0
    and sig0_shoulders[1] > 1e3 * sig0_star,
    "at k* the s channel switches off exactly — still scattering "
    "hard at k* − 1, recovered by orders of magnitude just past it: "
    "Ramsauer's mechanism, one phase recrossing π",
    f"sin²δ₀(k*) = {sin2_star:.1e}, σ₀ at k*−1 / k* / k*+0.4 = "
    f"{sig0_shoulders[0]:.2f} / {sig0_star:.1e} / "
    f"{sig0_shoulders[1]:.2f}",
)
validate.check(
    sig_peak / sig_min > 10.0,
    "and the well's total visibility still collapses more than "
    "tenfold from its low-energy peak — in argon, whose recrossing "
    "falls where s waves rule, that collapse is the measured minimum",
    f"peak/min = {sig_peak / sig_min:.1f}",
)

## Notebook summary

- The hard sphere set the scale: $\sigma \to 4\pi a^2$ at low energy
  and toward $2\pi a^2$ at high — a quantum cross-section is never
  the classical silhouette, and the partial-wave sum is as short as
  $\ell \lesssim ka$ promises.
- The numerical phase-shift extractor was certified three ways —
  hard-sphere channels to $10^{-8}$, the square well's closed form
  to $10^{-5}$, and the scattering length $2.09$ from a radius-1
  well.
- The optical theorem closed to eight decimals across three
  independent computations of $\sigma$, and the forward diffraction
  peak grew fifty-fold by $ka = 5$.
- Born was caught working (ratio $0.997$ at weak coupling) and caught
  failing (ratio $1.83$ at strong, monotonically worsening), and its
  $\mu \to 0$ limit reproduced Rutherford's formula to $10^{-13}$ —
  the classical result of [§2.5](../02-classical-mechanics/scattering.ipynb)
  recovered by first-order quantum mechanics.
- A well $5\%$ too shallow to bind a p state resonated instead:
  $\delta_1$ through $\pi/2$ at $E_R \approx 0.08$, Breit–Wigner
  shape fitted, width $\Gamma$ measured.
- And Levinson's theorem plus one phase passing through $\pi$
  explained Ramsauer's impossible transparent argon: the total
  cross-section collapsed more than tenfold at $k^*$.

## Outlook

- **Inelastic channels.** Real targets absorb as well as deflect;
  the phase shift becomes complex, $|S_\ell| < 1$, and the optical
  theorem accounts for everything the beam loses. That formalism is
  the daily language of nuclear and particle physics.
- **Resonances as poles.** $E_R - i\Gamma/2$ is a pole of the
  S-matrix in the complex energy plane — the rigorous version of
  Exercise 5, and the bridge to the decaying states of
  [§6.24](time-dependent-perturbation.ipynb).
- **Identical particles.** Scattering of indistinguishable particles
  ([§6.20](identical-particles.ipynb)) interferes the amplitude with
  its $\theta \to \pi - \theta$ mirror: bosons double the
  $90°$ rate, fermions extinguish it — one of the cleanest
  experimental signatures the symmetrization postulate has.
- **Cold atoms.** At microkelvin temperatures only $a_s$ survives
  (Exercise 2c), and tuning it with a magnetic field — the Feshbach
  resonance, Exercise 5's physics in a two-channel costume — is the
  control knob of every quantum-gas experiment, including the BEC of
  [§7.17](../07-quantum-statistical-mechanics/bose-einstein-condensation.ipynb).

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()